In [3]:
# Scenario
# A fast-growing e-commerce company called ShopEasy is struggling with inefficient marketing campaigns.
# Every day thousands of users visit their website. The marketing team spends a large amount of money showing ads, discounts,
# and promotional emails, but they don't know which customers are actually likely to buy something.
# Currently:
# Many customers browse but never purchase
# Marketing money is wasted on the wrong users
# The company wants to predict purchase probability
# The data science team has been asked to build a machine learning system that predicts whether a customer will purchase a product during a session.
# If the system predicts high probability of purchase, the system will:
# show personalized product recommendations
# offer targeted discounts
# prioritize marketing campaigns
# If the system predicts low probability, the company will avoid spending marketing resources.
# However, the dataset contains both numerical and categorical features, so the data science team must design a complete ML pipeline.
# Dataset is available in DatasetCapstoneProject3 in the github repo link https://github.com/himanshusar123/Datasets
# Business Objective
# Build a machine learning model that predicts whether a user will purchase (1) or not purchase (0) during a website session.
# The model must be implemented using scikit-learn pipelines, including:
# Encoding techniques
# Feature preprocessing
# Model training
# Model selection
# Hyperparameter tuning


import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_excel("DatasetCapstoneProject3.xlsx")

X = df.drop(["CustomerID","Purchased"], axis=1)
y = df["Purchased"]

numerical_features = ["Age","Time_on_Website","Pages_Visited","Ad_Clicks","Previous_Purchases"]
categorical_features = ["Gender","Device","Traffic_Source"]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression())
])

param_grid = [
    {
        "model":[LogisticRegression(max_iter=1000)],
        "model__C":[0.01,0.1,1,10]
    },
    {
        "model":[RandomForestClassifier()],
        "model__n_estimators":[50,100,200],
        "model__max_depth":[3,5,10]
    }
]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

grid = GridSearchCV(pipeline,param_grid,cv=5,scoring="accuracy")

grid.fit(X_train,y_train)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("Best Model:",grid.best_params_)
print("Accuracy:",accuracy_score(y_test,y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test,y_pred))
print("Classification Report:")
print(classification_report(y_test,y_pred))

Best Model: {'model': LogisticRegression(max_iter=1000), 'model__C': 1}
Accuracy: 1.0
Confusion Matrix:
[[1 0]
 [0 2]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3

